In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel('/Users/ahmedsallam/Documents/Measles_KSA/data/measles_2014_2023_master_audited.xlsx')

print("Dataset shape:", df.shape)
display (df.head())

print("Dataset shape:", df.shape)
display(df.head())

# Verify annual national case totals
year_cols = [col for col in df.columns if str(col).isdigit() and len(str(col)) == 4]

# Keep region rows; exclude totals and year-labelled rows from other tables.
regions = df["Region"].astype("string").str.strip()
wide = df.loc[
    regions.notna()
    & regions.str.lower().ne("total")
    & ~regions.str.fullmatch(r"\d{4}", na=False),
    ["Region", *year_cols],
].copy()

cases = wide.melt(
    id_vars="Region",
    var_name="year",
    value_name="reported_cases",
)
cases["year"] = cases["year"].astype(int)
cases["reported_cases"] = pd.to_numeric(cases["reported_cases"], errors="coerce")

df = cases.rename(columns={"Region": "health_region"})
annual_cases = df.groupby("year")["reported_cases"].sum()
display(annual_cases)

# Create a region-by-year incidence matrix
# Separate the case and population sections of the source table
population_marker = wide["Region"].astype("string").str.contains(
    "KSA population", case=False, na=False
)
marker_idx = wide.index[population_marker][0]

case_wide = wide.loc[wide.index < marker_idx].copy()
case_wide[year_cols] = case_wide[year_cols].apply(
    pd.to_numeric, errors="coerce"
)
case_wide = case_wide.loc[
    case_wide["Region"].notna()
    & case_wide["Region"].astype("string").str.lower().ne("total")
    & ~case_wide["Region"].astype("string").str.fullmatch(r"\d{4}", na=False)
].dropna(subset=year_cols, how="all")

population_wide = wide.loc[wide.index > marker_idx].copy()
population_wide[year_cols] = population_wide[year_cols].apply(
    pd.to_numeric, errors="coerce"
)
population_wide = population_wide.dropna(subset=year_cols, how="any")

# The population rows follow the case-region order in the source table.
if len(case_wide) != len(population_wide):
    raise ValueError("Could not align case and population region rows.")

population_wide["Region"] = case_wide["Region"].to_numpy()

cases_long = case_wide.melt(
    id_vars="Region", var_name="year", value_name="reported_cases"
)
population_long = population_wide.melt(
    id_vars="Region", var_name="year", value_name="population"
)

incidence = cases_long.merge(population_long, on=["Region", "year"])
incidence["annual_incidence_per_100k"] = (
    incidence["reported_cases"] / incidence["population"] * 100_000
)

incidence_matrix = incidence.pivot(
    index="Region",
    columns="year",
    values="annual_incidence_per_100k",
)

display(incidence_matrix)


Dataset shape: (76, 14)


,Region,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,Total,AAIR,Unnamed: 13
0,Riyadh,15,11,5,45,272,21,23,1,50,1157,1600,1.918064,NaN
1,Makkah,6,15,5,18,32,13,0,297,72,87,545,2.278438,NaN
2,Jeddah,52,11,3,65,231,17,7,14,22,67,489,1.083803,NaN
3,Ta`if,11,6,4,8,72,4,0,2,1,52,160,1.256368,NaN
4,Medinah,11,60,28,9,34,23,1,0,3,131,300,1.397055,NaN


Dataset shape: (76, 14)


,Region,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,Total,AAIR,Unnamed: 13
0,Riyadh,15,11,5,45,272,21,23,1,50,1157,1600,1.918064,NaN
1,Makkah,6,15,5,18,32,13,0,297,72,87,545,2.278438,NaN
2,Jeddah,52,11,3,65,231,17,7,14,22,67,489,1.083803,NaN
3,Ta`if,11,6,4,8,72,4,0,2,1,52,160,1.256368,NaN
4,Medinah,11,60,28,9,34,23,1,0,3,131,300,1.397055,NaN


year
2014    30770529.0
2015    31521637.0
2016    31742433.0
2017    32552793.0
2018    33414816.0
2019    33414684.0
2020    35013448.0
2021    34165908.0
2022    32175471.0
2023    32177478.0
Name: reported_cases, dtype: float64

year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
Region,,,,,,,,,,
Al-Ahsa,0.335184,0.081923,0.166929,0.244978,1.037581,0.159628,0.000000,0.000000,0.271673,2.082830
Al-Bahah,0.000000,0.211974,0.214416,0.210008,0.205293,0.205293,0.000000,0.000000,1.474170,1.769004
Al-Jouf,0.911087,0.890321,3.918342,1.474065,7.194369,146.765124,0.000000,0.000000,0.748492,1.247486
Aseer,0.113237,0.166075,0.280472,1.481734,6.976637,1.609993,0.000000,0.316738,0.466538,2.449327
Bishah,0.000000,0.000000,0.262150,0.000000,5.524085,0.000000,0.000000,0.000000,0.323073,0.323073
Eastern,0.761725,0.258487,0.318008,0.526599,2.169858,0.120548,0.028691,0.029514,0.731574,2.588647
Ha`il,0.149150,4.228515,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.535901,1.607704
Hafr Al-Baten,0.228650,0.000000,0.000000,0.000000,3.933137,0.437015,0.000000,0.000000,1.284777,1.927166
Jazan,0.195631,0.318730,0.456419,4.082812,4.988592,6.796956,0.119720,0.122198,0.711745,2.846981


In [2]:
import sys
import pandas as ps
import numpy as np
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Environment is ready!")


Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ]
Pandas: 2.2.2
NumPy: 1.26.4
Environment is ready!


In [3]:
from pathlib import Path
import pandas as pd

# Check the current working directory
print("Working directory:", Path.cwd())

# Locate the data folder
data_dir = Path("../data") if Path("../data").exists() else Path("data")

print("Data directory:", data_dir.resolve())

# Load the four datasets
regional = pd.read_csv(data_dir / "measles_region_year.csv")
monthly = pd.read_csv(data_dir / "measles_national_monthly.csv")
age = pd.read_csv(data_dir / "measles_national_age.csv")
annual = pd.read_csv(data_dir / "measles_national_annual.csv")

# Confirm that all datasets loaded
for name, df in {
    "Regional": regional,
    "Monthly": monthly,
    "Age": age,
    "Annual": annual
}.items():
    print(f"{name}: {df.shape[0]} rows × {df.shape[1]} columns")

Working directory: /Users/ahmedsallam/Documents/Measles_KSA/scripts
Data directory: /Users/ahmedsallam/Documents/Measles_KSA/data
Regional: 200 rows × 7 columns
Monthly: 120 rows × 6 columns
Age: 50 rows × 5 columns
Annual: 10 rows × 4 columns


In [4]:
#######################################
# Data Inspection and Quality Control#
#######################################

In [5]:

# Inspect all four datasets
datasets = {
    "Regional": regional,
    "Monthly": monthly,
    "Age": age,
    "Annual": annual
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"{name.upper()} DATASET")
    print(f"{'='*50}")

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

    print("\nFirst 3 rows:")
    display(df.head(3))



REGIONAL DATASET

Column names:
['health_region', 'year', 'reported_cases', 'population', 'annual_incidence_per_100k', 'region_total_cases_2014_2023', 'region_aair_per_100k_per_year']

Data types:
health_region                     object
year                               int64
reported_cases                     int64
population                         int64
annual_incidence_per_100k        float64
region_total_cases_2014_2023       int64
region_aair_per_100k_per_year    float64
dtype: object

Missing values:
health_region                    0
year                             0
reported_cases                   0
population                       0
annual_incidence_per_100k        0
region_total_cases_2014_2023     0
region_aair_per_100k_per_year    0
dtype: int64

Duplicate rows:
0

First 3 rows:


,health_region,year,reported_cases,population,annual_incidence_per_100k,region_total_cases_2014_2023,region_aair_per_100k_per_year
0,Riyadh,2014,15,7717467,0.194364,1600,1.918064
1,Riyadh,2015,11,7910864,0.139049,1600,1.918064
2,Riyadh,2016,5,8002100,0.062484,1600,1.918064



MONTHLY DATASET

Column names:
['year', 'month_number', 'month', 'reported_cases', 'national_annual_population', 'monthly_cases_per_100k_annual_population']

Data types:
year                                          int64
month_number                                  int64
month                                        object
reported_cases                                int64
national_annual_population                    int64
monthly_cases_per_100k_annual_population    float64
dtype: object

Missing values:
year                                        0
month_number                                0
month                                       0
reported_cases                              0
national_annual_population                  0
monthly_cases_per_100k_annual_population    0
dtype: int64

Duplicate rows:
0

First 3 rows:


,year,month_number,month,reported_cases,national_annual_population,monthly_cases_per_100k_annual_population
0,2014,1,Jan,17,30770375,0.055248
1,2014,2,Feb,20,30770375,0.064998
2,2014,3,Mar,17,30770375,0.055248



AGE DATASET

Column names:
['year', 'age_group', 'reported_cases', 'age_specific_population', 'age_specific_incidence_per_100k']

Data types:
year                                 int64
age_group                           object
reported_cases                       int64
age_specific_population            float64
age_specific_incidence_per_100k    float64
dtype: object

Missing values:
year                                0
age_group                           0
reported_cases                      0
age_specific_population            50
age_specific_incidence_per_100k    50
dtype: int64

Duplicate rows:
0

First 3 rows:


,year,age_group,reported_cases,age_specific_population,age_specific_incidence_per_100k
0,2014,lt1,33,NaN,NaN
1,2014,age1_lt5,55,NaN,NaN
2,2014,age5_lt15,26,NaN,NaN



ANNUAL DATASET

Column names:
['year', 'reported_cases', 'population', 'annual_crude_incidence_per_100k']

Data types:
year                                 int64
reported_cases                       int64
population                           int64
annual_crude_incidence_per_100k    float64
dtype: object

Missing values:
year                               0
reported_cases                     0
population                         0
annual_crude_incidence_per_100k    0
dtype: int64

Duplicate rows:
0

First 3 rows:


,year,reported_cases,population,annual_crude_incidence_per_100k
0,2014,154,30770375,0.500481
1,2015,219,31521418,0.694766
2,2016,125,31742308,0.393796


In [6]:
##########################################
# Numerical and epidemiological validation
##########################################

In [7]:

import numpy as np
import pandas as pd

# 1. Check unique observation identifiers
keys = {
    "Regional": (regional, ["health_region", "year"]),
    "Monthly": (monthly, ["year", "month_number"]),
    "Age": (age, ["year", "age_group"]),
    "Annual": (annual, ["year"])
}

for name, (df, columns) in keys.items():
    duplicates = df.duplicated(subset=columns).sum()
    print(f"{name} duplicate identifiers: {duplicates}")

# 2. Reconcile case totals for every year
comparison = pd.DataFrame({
    "regional": regional.groupby("year")["reported_cases"].sum(),
    "monthly": monthly.groupby("year")["reported_cases"].sum(),
    "age": age.groupby("year")["reported_cases"].sum(),
    "national": annual.set_index("year")["reported_cases"]
})

comparison["cases_match"] = comparison.nunique(axis=1) == 1
print("\nCase reconciliation:")
display(comparison)

# 3. Check population totals
regional_pop = regional.groupby("year")["population"].sum()
national_pop = annual.set_index("year")["population"]

print("\nPopulation totals match:",
      regional_pop.equals(national_pop))

# 4. Recalculate annual incidence
expected_incidence = (
    regional["reported_cases"] / regional["population"] * 100000
)

print("\nAnnual regional incidence matches:",
      np.allclose(
          regional["annual_incidence_per_100k"],
          expected_incidence,
          rtol=1e-6,
          atol=1e-4
      ))

# 5. Recalculate AAIR for each health region
region_summary = regional.groupby("health_region").agg(
    total_cases=("reported_cases", "sum"),
    population_exposure=("population", "sum"),
    reported_aair=("region_aair_per_100k_per_year", "first")
)

region_summary["calculated_aair"] = (
    region_summary["total_cases"]
    / region_summary["population_exposure"]
    * 100000
)

region_summary["aair_matches"] = np.isclose(
    region_summary["reported_aair"],
    region_summary["calculated_aair"],
    rtol=1e-6,
    atol=0.01
)

print("\nAAIR validation:")
display(region_summary)

print("\nAll regional AAIR values match:",
      region_summary["aair_matches"].all())


Regional duplicate identifiers: 0
Monthly duplicate identifiers: 0
Age duplicate identifiers: 0
Annual duplicate identifiers: 0

Case reconciliation:


,regional,monthly,age,national,cases_match
year,,,,,
2014,154,154,154,154,True
2015,219,219,219,219,True
2016,125,125,125,125,True
2017,457,457,457,457,True
2018,1156,1156,1156,1156,True
2019,1024,1024,1024,1024,True
2020,34,34,34,34,True
2021,327,327,327,327,True
2022,247,247,247,247,True



Population totals match: True

Annual regional incidence matches: True

AAIR validation:


,total_cases,population_exposure,reported_aair,calculated_aair,aair_matches
health_region,,,,,
Al-Ahsa,51,12132673,0.420353,0.420353,True
Al-Bahah,16,4531270,0.353102,0.353102,True
Al-Jouf,567,3551052,15.967099,15.967099,True
Aseer,253,18168190,1.392544,1.392544,True
Bishah,25,3775568,0.662152,0.662152,True
Eastern,254,33105189,0.767251,0.767251,True
Ha`il,46,7143656,0.643928,0.643928,True
Hafr Al-Baten,36,4563071,0.788942,0.788942,True
Jazan,322,15528022,2.073670,2.073670,True



All regional AAIR values match: True


In [11]:

# Aggregate regional observed and expected cases
oe_summary = (
    oe_data
    .groupby("health_region", as_index=False)
    .agg(
        observed_cases=("reported_cases", "sum"),
        expected_cases=("expected_cases", "sum"),
        population_exposure=("population", "sum")
    )
)

# Calculate observed-to-expected ratio
oe_summary["oe_ratio"] = (
    oe_summary["observed_cases"]
    / oe_summary["expected_cases"]
)

# Sort by O/E ratio
oe_summary = oe_summary.sort_values(
    "oe_ratio",
    ascending=False
)

display(oe_summary)


,health_region,observed_cases,expected_cases,population_exposure,oe_ratio
16,Qurayyat,317,33.364954,1763900,9.500987
2,Al-Jouf,567,67.630506,3551052,8.383791
12,Najran,603,108.626582,5906522,5.551127
10,Makkah,545,456.721932,23919901,1.193286
8,Jazan,322,277.745593,15528022,1.159334
17,Riyadh,1600,1550.510950,83417447,1.031918
13,Northern,55,68.464746,3722828,0.803333
14,Qaseem,204,256.653940,14181605,0.794845
3,Aseer,253,328.866200,18168190,0.769310
11,Medinah,300,395.169558,21473746,0.759168


In [12]:

import numpy as np

print("Total observed:", oe_summary["observed_cases"].sum())
print("Total expected:", oe_summary["expected_cases"].sum())

print(
    "National reconciliation:",
    np.isclose(
        oe_summary["observed_cases"].sum(),
        oe_summary["expected_cases"].sum()
    )
)

print(
    "Missing O/E values:",
    oe_summary["oe_ratio"].isna().sum()
)


Total observed: 5997
Total expected: 5997.0
National reconciliation: True
Missing O/E values: 0
